# Experiment - HEST cohort filtering and sample-level split

This notebook explains the alternate filtering and split from `HEST_Patient_Level_Split_EN.ipynb`. The saved preview run filtered 108 breast Spatial Transcriptomics samples and assigned 75 to train and 33 to test. It exported 100 paired spots from one sample per partition. These were structural checks, not a trained-model result.

The grouping unit was sample ID. If a patient contributes multiple samples, this does **not** guarantee patient independence; the onboarding ridge evaluation uses patient groups for its primary estimate. This notebook calls the shared metadata filter in `src/`.


## 1. Find the repository and install its project dependencies
This block prepares the import path for shared HEST helpers.

In [ ]:
from pathlib import Path
import sys, subprocess
_here = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (_here, *_here.parents) if (p/'requirements.txt').is_file()), None)
if PROJECT_ROOT is None: raise RuntimeError('Run inside the Hist2ST repository.')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT/'requirements.txt')])
sys.path.insert(0, str(PROJECT_ROOT/'src'))


## 2. Query and filter the catalog
The reusable function checks exact category choices and prints available categories before returning selected IDs.

In [ ]:
from huggingface_hub import login
from hest_download_pipeline import prepare_selection
login()
ids = prepare_selection(organs=['Breast'], species=['Homo sapiens'],
                        st_technologies=['Spatial Transcriptomics'],
                        metadata_filename='HEST_v1_3_0.csv')
print('Selected samples:', len(ids))


## 3. Split selected sample IDs
This reproduces the historical 70/30 sample-level design. The 75/33 count is the recorded source-run result; a current catalog revision or split implementation can produce different IDs/counts.

In [ ]:
import random
selected = sorted(ids)
random.Random(42).shuffle(selected)
n_test = round(len(selected) * 0.30)
test_ids, train_ids = selected[:n_test], selected[n_test:]
print('Current demonstration:', len(train_ids), 'train samples;', len(test_ids), 'test samples')
print('ID overlap:', bool(set(train_ids) & set(test_ids)))


## 4. Optional small download
The downloader can fetch selected sample data. Keep this off for the metadata-only walkthrough; HEST files can be large. The original preview downloaded one sample per partition and exported at most 100 matched spots.

In [ ]:
from hest_download_pipeline import download_selection
DOWNLOAD_PREVIEW = False
PREVIEW_IDS = [train_ids[0], test_ids[0]]
if DOWNLOAD_PREVIEW:
    destination = PROJECT_ROOT / 'data' / 'sample_split_preview'
    download_selection(PREVIEW_IDS, output_dir=destination, data_folders=['patches','st'])
    print('Downloaded preview to:', destination)
else:
    print('Preview download skipped. Set DOWNLOAD_PREVIEW=True to fetch two samples.')


## 5. Interpret
A disjoint sample-ID split is not the same as a disjoint-patient split. Before reporting an independent patient result, compare patient IDs across partitions and move all sections from one patient together. The preview?s successful file matching confirms structure only, not biological validity or predictive accuracy.